# 25c — 2D Gaussian-summary optimizer (mean + spread)

Optimize (μ, γ) with the 2D Gaussian-summary loss: match the mean and covariance of
(FWHM, σ_fit) to the data. γ is updated pathwise (implicit derivatives); μ by the count
score with 2D-Gaussian responsibilities. No bandwidth. Robust tanh-saturated update,
fixed init (μ=50, γ=10), fresh noise per step.

Result: the optimizer is BIASED — it does not recover the truth. mu overshoots to 1.9-4.2x mu_true at low/mid transmission and undershoots (0.5-0.6x) at 3nW high T; gamma overshoots to 2.2-3.8x gamma_true at 1nW. This confirms the pre-run diagnostic: matching mean+covariance of the heavy-tailed (FWHM, sigma_fit) summary is a biased estimator — the tail inflates the data spread and the optimizer inflates mu/gamma chasing it. The animated figures show the (mu, gamma) path and the cloud-vs-data at every step.

In [1]:
import os, sys, time
for _up in ('', '..', '../..'):
    _p = os.path.abspath(_up)
    if os.path.exists(os.path.join(_p, 'src', 'fisher.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break

import numpy as np
import torch

from src.fisher import (EXPERIMENTS, gaussian_summary, load_real_target, make_pool, sim_cloud)

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.ndimage import gaussian_filter
from IPython.display import display

pio.templates.default = 'plotly_white'

M_FINAL = 100      # cloud size per optimizer step
N_ITER  = 40       # optimization steps per experiment
MU_INIT, GAM_INIT = 50.0, 10.0
LR, G_M, G_G = 0.03, 10.0, 10.0
SEED = 42
N_WORKERS = os.cpu_count() or 4
torch.set_num_threads(1)
print('ready —', len(EXPERIMENTS), 'experiments,', N_WORKERS, 'workers,', N_ITER, 'iters')

ready — 14 experiments, 4 workers, 40 iters


In [2]:
results = []
t0 = time.time()
with make_pool(N_WORKERS) as pool:
    for (power, trans, mu_true, sigma_prop, lam, gamma_true) in EXPERIMENTS:
        target_f, target_s = load_real_target(int(power.replace('nW', '')), trans)
        mu, gam = MU_INIT, GAM_INIT
        history = []
        for it in range(N_ITER + 1):
            cloud = sim_cloud(mu, sigma_prop, lam, gam, M_FINAL, SEED + it, pool=pool)
            r = gaussian_summary(cloud, target_f, target_s, mu, sigma_prop)
            history.append(dict(step=it, mu=mu, gamma=gam,
                                cloud_f=cloud[0].numpy().astype(np.float32),
                                cloud_s=cloud[1].numpy().astype(np.float32)))
            if it < N_ITER:
                gam = max(gam * (1.0 - LR * np.tanh(r['grad_gamma'] / G_G)), 0.5)
                mu  = max(mu  * (1.0 - LR * np.tanh(r['grad_mu']    / G_M)), 0.5)
        results.append(dict(power=power, T=trans, exp=f"{power}T{trans}",
                            mu_init=MU_INIT, gamma_init=GAM_INIT,
                            mu_true=mu_true, gamma_true=gamma_true,
                            target_f=target_f.numpy().astype(np.float32),
                            target_s=target_s.numpy().astype(np.float32),
                            history=history))
        h = history[-1]
        print(f"[{power} T{trans}] mu {mu_true:7.2f} -> {h['mu']:7.2f} | gam {gamma_true:5.2f} -> {h['gamma']:6.2f}  ({time.time()-t0:.0f}s)", flush=True)

print(f"done in {time.time()-t0:.0f}s — {len(results)} rows")

[1nW T5] mu    9.39 ->   30.48 | gam  8.50 ->  32.62  (78s)


[1nW T10] mu   12.37 ->   24.31 | gam  8.50 ->  30.17  (153s)


[1nW T20] mu   17.32 ->   33.23 | gam  8.50 ->  21.48  (227s)


[1nW T40] mu   38.41 ->   98.97 | gam  8.50 ->  30.35  (303s)


[1nW T60] mu   61.37 ->  163.10 | gam  8.50 ->  19.60  (378s)


[1nW T80] mu   79.36 ->  163.10 | gam  8.50 ->  20.07  (453s)


... (5 outputs left out)

[3nW T60] mu  103.20 ->  130.30 | gam 14.10 ->  18.71  (912s)


[3nW T80] mu  137.54 ->   82.56 | gam 14.10 ->  11.52  (987s)


[3nW T100] mu  175.71 ->   90.07 | gam 14.10 ->  11.07  (1061s)


done in 1061s — 14 rows


In [3]:
def _grid_range(f, s, pct=(2, 98), pad=0.12):
    lo_f, hi_f = np.percentile(f, pct); lo_s, hi_s = np.percentile(s, pct)
    df, ds = max(hi_f - lo_f, 1e-6), max(hi_s - lo_s, 1e-6)
    return (lo_f - pad*df, hi_f + pad*df), (lo_s - pad*ds, hi_s + pad*ds)

def _density(cf, cs, xr, yr, nx=36, ny=36, smooth=1.2):
    H, xe, ye = np.histogram2d(cf, cs, bins=[nx, ny], range=[xr, yr])
    Z = gaussian_filter(H.T, smooth)
    Z = Z / max(Z.max(), 1e-12)
    return 0.5*(xe[:-1]+xe[1:]), 0.5*(ye[:-1]+ye[1:]), np.round(Z*100).astype(int)

def build_power_figure(results, power):
    rs = sorted([r for r in results if r['power'] == power], key=lambda r: r['T'])
    if not rs:
        print(f'no {power} experiments -> figure skipped'); return None

    grids, pranges = {}, {}
    for r in rs:
        cf = np.concatenate([fr['cloud_f'] for fr in r['history']])
        cs = np.concatenate([fr['cloud_s'] for fr in r['history']])
        grids[r['exp']] = _grid_range(np.concatenate([cf, r['target_f']]),
                                      np.concatenate([cs, r['target_s']]))
        pmus = [fr['mu'] for fr in r['history']] + [r['mu_true'], r['mu_init']]
        pgam = [fr['gamma'] for fr in r['history']] + [r['gamma_true'], r['gamma_init']]
        pranges[r['exp']] = ((0, max(pmus)*1.05), (0, max(pgam)*1.05))

    fig = make_subplots(rows=1, cols=2, column_widths=[0.40, 0.60], horizontal_spacing=0.10,
                        subplot_titles=['(μ, γ) path', '(FWHM, σ_fit)  simulated cloud  vs  real data'])
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dict(color='#1d3557', width=2), name='path'), 1, 1)
    fig.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(color='#e63946', size=12, symbol='x'), name='current'), 1, 1)
    fig.add_trace(go.Contour(x=[0,1], y=[0,1], z=[[0,0],[0,0]], colorscale='Blues', showscale=False,
                             contours=dict(showlines=False), opacity=0.85, name='sim density'), 1, 2)
    fig.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(color='#457b9d', size=4, opacity=0.45), name='sim draws'), 1, 2)
    fig.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(color='#2a9d8f', size=11, symbol='circle-open',
                                                                     line=dict(width=2)), name='init 0.5x'), 1, 1)
    fig.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(color='#000000', size=13, symbol='cross'), name='truth'), 1, 1)
    fig.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(color='#e63946', size=5, opacity=0.85,
                                                                     symbol='diamond'), name='real data'), 1, 2)

    frames = []
    for r in rs:
        xr, yr = grids[r['exp']]; (px0, px1), (py0, py1) = pranges[r['exp']]
        for fr in r['history']:
            k = fr['step']
            px = [h['mu'] for h in r['history'][:k+1]]
            py = [h['gamma'] for h in r['history'][:k+1]]
            xc, yc, Z = _density(fr['cloud_f'], fr['cloud_s'], xr, yr)
            data = [go.Scatter(x=px, y=py),
                    go.Scatter(x=[px[-1]], y=[py[-1]]),
                    go.Contour(x=xc, y=yc, z=Z),
                    go.Scatter(x=fr['cloud_f'], y=fr['cloud_s'])]
            if k == 0:
                data += [go.Scatter(x=[r['mu_init']], y=[r['gamma_init']]),
                         go.Scatter(x=[r['mu_true']], y=[r['gamma_true']]),
                         go.Scatter(x=r['target_f'], y=r['target_s'])]
            frames.append(go.Frame(name=f"{r['exp']}|{k}", data=data, layout=go.Layout(
                title_text=f"{r['exp']}  —  step {k}/{N_ITER}   (μ x{px[-1]/r['mu_true']:.2f}, γ x{py[-1]/r['gamma_true']:.2f})",
                xaxis=dict(range=[px0, px1], title='μ (photons)'),
                yaxis=dict(range=[py0, py1], title='γ (MHz)'),
                xaxis2=dict(range=[xr[0], xr[1]], title='FWHM (MHz)'),
                yaxis2=dict(range=[yr[0], yr[1]], title='σ_fit (MHz)'),
            )))
    fig.frames = frames

    for tr, d in zip(fig.data, frames[0].data):
        for prop in ('x', 'y', 'z'):
            val = getattr(d, prop, None)
            if val is not None:
                setattr(tr, prop, val)

    exps = [r['exp'] for r in rs]
    play  = dict(label='▶ play', method='animate',
                 args=[None, dict(frame=dict(duration=350, redraw=True), fromcurrent=True, transition=dict(duration=0))])
    pause = dict(label='❚❚ pause', method='animate', args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])
    dropdown = [dict(label=e, method='animate',
                     args=[[f'{e}|0'], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
                for e in exps]
    steps = [dict(method='animate', label=(e if k == 0 else str(k)),
                  args=[[f'{e}|{k}'], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
             for e in exps for k in range(N_ITER + 1)]
    fig.update_layout(
        height=620, margin=dict(t=120, b=40),
        updatemenus=[
            dict(type='buttons', direction='left', x=0.0, y=1.18, xanchor='left', buttons=[play, pause]),
            dict(type='dropdown', direction='down', x=1.0, y=1.18, xanchor='right',
                 showactive=True, buttons=dropdown, active=0),
        ],
        sliders=[dict(active=0, x=0.0, len=1.0, y=-0.06, currentvalue=dict(prefix=''), steps=steps)],
    )
    return fig

for power in ('1nW', '3nW'):
    fig = build_power_figure(results, power)
    if fig is not None:
        display({'application/vnd.plotly.v1+json': pio.to_json(fig)}, raw=True)